# JFDS Experiment

In [ ]:
import numpy as np
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import curve_fit
from itertools import combinations
from IPython.display import display, Markdown
from sklearn.metrics.pairwise import cosine_similarity
import optuna
import numpy as np
import pandas as pd
optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR           = Path("../dataset/movie_lens")
RANDOM_SEED        = 42
TEST_FRACTION      = 0.20
N_FACTORS          = 20
SVD_EPOCHS         = 12
BPR_EPOCHS         = 12
LEARNING_RATE      = 0.01
REG                = 0.02
BATCH_SIZE         = 8192
POOL_SIZE          = 50
K                  = 10
LAMBDA_F_DEFAULT   = 0.25
LAMBDA_D_DEFAULT   = 0.25
ANALYSIS_SAMPLE_N  = 1500

RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams['figure.dpi'] = 110
C_TOPK, C_JFDS, C_SVD, C_BPR = '#3498DB', '#E74C3C', '#9B59B6', '#16A085'
print('Config loaded.')


---
## 1. Load MovieLens 1M

In [ ]:
movies_df = pd.read_csv(DATA_DIR / 'movies.dat', sep='::', engine='python',
                        encoding='latin-1', header=None,
                        names=['MovieID', 'Title', 'Genres'])

users_df = pd.read_csv(DATA_DIR / 'users.dat', sep='::', engine='python',
                       encoding='latin-1', header=None,
                       names=['UserID', 'Gender', 'Age', 'Occupation', 'Zip'])

ratings_df = pd.read_csv(DATA_DIR / 'ratings.dat', sep='::', engine='python',
                         encoding='latin-1', header=None,
                         names=['UserID', 'MovieID', 'Rating', 'Timestamp'])

print(f"movies : {len(movies_df):,}   users : {len(users_df):,}   ratings : {len(ratings_df):,}")
movies_df.head()


In [ ]:
# Contiguous ID remapping + genre vectors
unique_users  = np.sort(ratings_df['UserID'].unique())
unique_movies = np.sort(movies_df['MovieID'].unique())
user2idx  = {u: i for i, u in enumerate(unique_users)}
movie2idx = {m: i for i, m in enumerate(unique_movies)}
N_USERS, N_ITEMS = len(unique_users), len(unique_movies)

ratings_df['u_idx'] = ratings_df['UserID'].map(user2idx)
ratings_df['i_idx'] = ratings_df['MovieID'].map(movie2idx)
ratings_df = ratings_df.dropna(subset=['i_idx']).copy()
ratings_df['i_idx'] = ratings_df['i_idx'].astype(int)

movies_df = movies_df.set_index('MovieID').loc[unique_movies].reset_index()
all_genres = sorted({g for gl in movies_df['Genres'] for g in gl.split('|')})
N_GENRES = len(all_genres)
genre2idx = {g: i for i, g in enumerate(all_genres)}

genre_vec = np.zeros((N_ITEMS, N_GENRES))
for _, row in movies_df.iterrows():
    idx = movie2idx[row['MovieID']]
    for g in row['Genres'].split('|'):
        genre_vec[idx, genre2idx[g]] = 1.0
genre_vec = genre_vec / genre_vec.sum(axis=1, keepdims=True)

print("Pre-computing genre similarity matrix ...")
GENRE_SIM = cosine_similarity(genre_vec).astype(np.float32)
print(f"GENRE_SIM shape: {GENRE_SIM.shape}  dtype: {GENRE_SIM.dtype}")

users_df['u_idx'] = users_df['UserID'].map(user2idx)
users_df = users_df.dropna(subset=['u_idx']).copy()
users_df['u_idx'] = users_df['u_idx'].astype(int)

print(f"N_USERS={N_USERS}   N_ITEMS={N_ITEMS}   N_GENRES={N_GENRES}")
print('Genres:', all_genres)


## Data Preprocessing

This step prepares the dataset for model training by converting the original IDs into contiguous indices and generating genre based representations for movies.

### Steps

1. **Remap User and Movie IDs**
   - Convert original `UserID` and `MovieID` values into contiguous integer indices (`u_idx`, `i_idx`).
   - This makes embedding lookups efficient and compatible with deep learning frameworks.

2. **Filter and Align Data**
   - Remove ratings for movies that are not present in the movie metadata.
   - Reorder the movie dataset so that its rows match the new movie indices.

3. **Create Genre Vectors**
   - Extract all unique genres.
   - Represent each movie as a multi-hot vector indicating its genres.
   - Normalize each genre vector so its values sum to 1.

4. **Compute Genre Similarity**
   - Calculate a cosine similarity matrix between all movie genre vectors.
   - This matrix captures semantic similarity between movies based on shared genres and can be used for genre-aware recommendations or regularization.

5. **Remap User Metadata**
   - Apply the same user index mapping to the user metadata table to maintain consistency across all datasets.

### Output

- `u_idx`: Contiguous user indices.
- `i_idx`: Contiguous movie indices.
- `genre_vec`: Normalized genre representation for each movie.
- `GENRE_SIM`: Movie-to-movie genre similarity matrix.
- `N_USERS`, `N_ITEMS`, `N_GENRES`: Dataset statistics.

---
## 2. Train / Test Split

In [ ]:
ratings_df = ratings_df.sort_values(['UserID', 'Timestamp']).reset_index(drop=True)
ratings_df['rank_desc']  = ratings_df.groupby('UserID').cumcount(ascending=False)
ratings_df['user_count'] = ratings_df.groupby('UserID')['UserID'].transform('count')
test_cutoff = np.maximum(1, (ratings_df['user_count'] * TEST_FRACTION).astype(int))
test_mask = ratings_df['rank_desc'] < test_cutoff

train_df = ratings_df.loc[~test_mask, ['UserID','MovieID','Rating','Timestamp','u_idx','i_idx']].reset_index(drop=True)
test_df  = ratings_df.loc[ test_mask, ['UserID','MovieID','Rating','Timestamp','u_idx','i_idx']].reset_index(drop=True)

train_seen    = train_df.groupby('u_idx')['i_idx'].apply(set).to_dict()
test_relevant = test_df[test_df['Rating'] >= 4].groupby('u_idx')['i_idx'].apply(set).to_dict()
test_grades   = test_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()

print(f"train ratings: {len(train_df):,}   test ratings: {len(test_df):,}")
print(f"users with >=1 relevant test item: {len(test_relevant):,} / {N_USERS}")


In [ ]:
# Popularity tiers from TRAIN only
pop_count = np.zeros(N_ITEMS)
counts = train_df['i_idx'].value_counts()
pop_count[counts.index.values] = counts.values
pop_norm = pop_count / (pop_count.max() + 1e-12)

order_by_pop = np.argsort(-pop_count)
tier = np.empty(N_ITEMS, dtype=object)
tier[order_by_pop[:int(.2 * N_ITEMS)]]                  = 'head'
tier[order_by_pop[int(.2 * N_ITEMS):int(.5 * N_ITEMS)]] = 'mid'
tier[order_by_pop[int(.5 * N_ITEMS):]]                  = 'tail'
TIERS = ['head', 'mid', 'tail']
TARGET_SHARE = {t: 1/3 for t in TIERS}

print(pd.Series(tier).value_counts())


## Popularity Tier Assignment

Movies are grouped into popularity tiers using **only the training data**.

- Count interactions for each movie.
- Rank movies by popularity.
- Assign tiers:
  - **Head:** Top 20%
  - **Mid:** Next 30%
  - **Tail:** Remaining 50%
- Define an equal target recommendation share (33.3%) for each tier.

---
## 3. Base Recommenders: SVD and BPR 


In [ ]:
def train_svd(train_df, n_users, n_items, n_factors=N_FACTORS, n_epochs=SVD_EPOCHS,
              lr=LEARNING_RATE, reg=REG, batch_size=BATCH_SIZE, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    global_mean = train_df['Rating'].mean()
    b_u = np.zeros(n_users)
    b_i = np.zeros(n_items)
    P   = rng.normal(0, 0.1, (n_users, n_factors))
    Q   = rng.normal(0, 0.1, (n_items, n_factors))

    u_idx = train_df['u_idx'].values
    i_idx = train_df['i_idx'].values
    r_val = train_df['Rating'].values.astype(float)
    n = len(train_df)

    for epoch in range(n_epochs):
        perm = rng.permutation(n)
        for start in range(0, n, batch_size):
            b  = perm[start:start + batch_size]
            bu, bi, rv = u_idx[b], i_idx[b], r_val[b]

            pred = global_mean + b_u[bu] + b_i[bi] + np.einsum('bf,bf->b', P[bu], Q[bi])
            err  = rv - pred

            # ⚡ bincount scatter-add (replaces slow np.add.at)
            b_u += lr * (np.bincount(bu, weights=err - reg * b_u[bu], minlength=n_users))
            b_i += lr * (np.bincount(bi, weights=err - reg * b_i[bi], minlength=n_items))

            # Factor updates: accumulate per-index gradients then add once
            err_col = err[:, None]  # (B,1)
            P_grad = err_col * Q[bi] - reg * P[bu]   # (B, F)
            Q_grad = err_col * P[bu] - reg * Q[bi]   # (B, F)
            for f in range(n_factors):
                P[:, f] += lr * np.bincount(bu, weights=P_grad[:, f], minlength=n_users)
                Q[:, f] += lr * np.bincount(bi, weights=Q_grad[:, f], minlength=n_items)

        pred_all = global_mean + b_u[u_idx] + b_i[i_idx] + np.einsum('nf,nf->n', P[u_idx], Q[i_idx])
        rmse = np.sqrt(np.mean((r_val - pred_all) ** 2))
        print(f"  [SVD] epoch {epoch+1:>2}/{n_epochs}  train RMSE={rmse:.4f}")

    return {'global_mean': global_mean, 'b_u': b_u, 'b_i': b_i, 'P': P, 'Q': Q}

def svd_score_user(model, u):
    return model['global_mean'] + model['b_u'][u] + model['b_i'] + model['Q'] @ model['P'][u]


In [ ]:
print('Training SVD ...')
svd_model = train_svd(train_df, N_USERS, N_ITEMS)
print('SVD training complete.')


In [ ]:
def train_bpr(train_df, n_users, n_items, n_factors=N_FACTORS, n_epochs=BPR_EPOCHS,
              lr=LEARNING_RATE, reg=REG, batch_size=BATCH_SIZE, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    P   = rng.normal(0, 0.1, (n_users, n_factors))
    Q   = rng.normal(0, 0.1, (n_items, n_factors))
    b_i = np.zeros(n_items)

    pos_df    = train_df[train_df['Rating'] >= 4]
    pos_users = pos_df['u_idx'].values
    pos_items = pos_df['i_idx'].values
    n_pos = len(pos_users)

    for epoch in range(n_epochs):
        perm = rng.permutation(n_pos)
        for start in range(0, n_pos, batch_size):
            b  = perm[start:start + batch_size]
            bu, bi = pos_users[b], pos_items[b]
            bj = rng.integers(0, n_items, size=len(b))

            x_uij = np.einsum('bf,bf->b', P[bu], Q[bi] - Q[bj]) + b_i[bi] - b_i[bj]
            sig   = 1.0 / (1.0 + np.exp(np.clip(x_uij, -30, 30)))

            # ⚡ bincount scatter-add
            dQ = Q[bi] - Q[bj]

            for f in range(n_factors):
                P[:, f]  += lr * (np.bincount(bu, weights=sig * dQ[:, f],    minlength=n_users)
                                 - reg * np.bincount(bu, weights=P[bu, f], minlength=n_users))
                Q[:, f]  += lr * (np.bincount(bi, weights= sig * P[bu, f],   minlength=n_items)
                                 - reg * np.bincount(bi, weights=Q[bi, f], minlength=n_items))
                Q[:, f]  += lr * (np.bincount(bj, weights=-sig * P[bu, f],   minlength=n_items)
                                 - reg * np.bincount(bj, weights=Q[bj, f], minlength=n_items))

            b_i += lr * (np.bincount(bi, weights= sig - reg * b_i[bi], minlength=n_items)
                       + np.bincount(bj, weights=-sig - reg * b_i[bj], minlength=n_items))

        print(f"  [BPR] epoch {epoch+1:>2}/{n_epochs} complete")

    return {'P': P, 'Q': Q, 'b_i': b_i}

def bpr_score_user(model, u):
    return model['Q'] @ model['P'][u] + model['b_i']

In [ ]:
print('Training BPR ...')
bpr_model = train_bpr(train_df, N_USERS, N_ITEMS)
print('BPR training complete.')


In [ ]:
def build_score_matrix_svd(model):
    return (model['global_mean']
            + model['b_u'][:, None]
            + model['b_i'][None, :]
            + model['P'] @ model['Q'].T)   

def build_score_matrix_bpr(model):
    return model['P'] @ model['Q'].T + model['b_i'][None, :]

def build_candidates_fast(score_matrix, seen_dict, pool_size=POOL_SIZE):
    pools = {}
    for u in range(score_matrix.shape[0]):
        scores = score_matrix[u].copy()
        seen   = seen_dict.get(u, set())
        if seen:
            scores[list(seen)] = -np.inf
        top_idx = np.argpartition(scores, -pool_size)[-pool_size:]
        top_idx = top_idx[np.argsort(-scores[top_idx])]
        pools[u] = (top_idx, scores[top_idx])
    return pools

print('Building SVD score matrix ...')
svd_score_mat = build_score_matrix_svd(svd_model)
print('Building BPR score matrix ...')
bpr_score_mat = build_score_matrix_bpr(bpr_model)

print('Building candidate pools ...')
svd_pools = build_candidates_fast(svd_score_mat, train_seen)
bpr_pools = build_candidates_fast(bpr_score_mat, train_seen)
print(f"SVD pools: {len(svd_pools)}   BPR pools: {len(bpr_pools)}")


## Candidate Pool Generation

Candidate pools are created for both the SVD and BPR models.

- Compute recommendation scores for every user movie pair.
- Remove movies that users have already interacted with.
- Select the top scoring unseen movies as candidate recommendations.
- Store these candidate pools for efficient recommendation and evaluation.

---
## 4a. Rerankers

In [ ]:
# ──────────────────────────────────────────────────────────────
# Generic greedy reranking engine + Top-K baseline
# ──────────────────────────────────────────────────────────────
def greedy_rerank(cand_ids, cand_rel, score_fn, k=K, **params):
    rel_norm = (cand_rel - cand_rel.min()) / (cand_rel.max() - cand_rel.min() + 1e-12)
    rel_map  = dict(zip(cand_ids, rel_norm))
    remaining = list(cand_ids)
    selected, tier_counts = [], {t: 0 for t in TIERS}

    for step in range(min(k, len(cand_ids))):
        best_score, best_item = -np.inf, None
        for c in remaining:
            s = score_fn(c, rel_map[c], selected, tier_counts, step, **params)
            if s > best_score:
                best_score, best_item = s, c
        selected.append(best_item)
        tier_counts[tier[best_item]] += 1
        remaining.remove(best_item)
    return selected

def topk_rerank(cand_ids, cand_rel, k=K):
    order = np.argsort(-cand_rel)
    return list(np.array(cand_ids)[order[:k]])


### JFDS EQUATION

In [ ]:
def fair_boost(cand_idx, tier_counts, n_selected):
    t = tier[cand_idx]
    current_share = tier_counts[t] / max(1, n_selected)
    gap = max(0.0, TARGET_SHARE[t] - current_share)
    return (gap ** 2) / (TARGET_SHARE[t] ** 2)

def diversity_term_fast(cand_idx, selected_idx):
    """⚡ Uses pre-computed GENRE_SIM — no per-call cosine computation."""
    if not selected_idx:
        return 1.0
    return float(1.0 - GENRE_SIM[cand_idx, selected_idx].mean())

def jfds_score(cand_idx, rel_value, selected, tier_counts, step,
               lam_f=LAMBDA_F_DEFAULT, lam_d=LAMBDA_D_DEFAULT):
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def jfds_rerank(cand_ids, cand_rel, lam_f=LAMBDA_F_DEFAULT, lam_d=LAMBDA_D_DEFAULT, k=K):
    return greedy_rerank(cand_ids, cand_rel, jfds_score, k=k, lam_f=lam_f, lam_d=lam_d)


---
### Established Baselines: MMR and Fairness-Only

To make the case for JFDS credible we compare it against two established,
**single-objective** re-ranking methods that each isolate one half of what
JFDS does jointly. Both reuse the same `greedy_rerank` engine and the same
`GENRE_SIM` / `tier` data structures as JFDS, so every strategy sees an
identical candidate pool and an identical notion of "diversity" and
"fairness" — a fair head-to-head comparison, not an apples-to-oranges one.

- **MMR (Carbonell & Goldstein, 1998)** — diversity only, no fairness term:
  $$\text{score}(i,S) = \lambda \cdot rel(i) - (1-\lambda)\max_{j \in S} sim(i,j)$$
- **Fairness-only** — exposure-fairness only, no diversity term:
  $$\text{score}(i) = (1-\lambda_f)\cdot rel(i) + \lambda_f \cdot fair\_boost(i)$$

If JFDS outperforms **both** specialists on the metric each was designed to
win — without giving up materially more relevance — that is much stronger
evidence than beating a no-objective baseline (Top-K) alone.


In [ ]:
MMR_LAMBDA        = 0.7    # Strategy C (MMR) — fixed relevance weight, standard literature value
FAIR_ONLY_LAMBDA  = 0.45   # Strategy D (fairness-only) — fixed fairness weight, matches JFDS's optimizable upper range

# ── Strategy C: MMR — diversity only, no fairness term ──────────────────────
def mmr_max_sim(cand_idx, selected_idx):
    """Max genre-cosine similarity between a candidate and everything already selected."""
    if not selected_idx:
        return 0.0
    return float(GENRE_SIM[cand_idx, selected_idx].max())

def mmr_score(cand_idx, rel_value, selected, tier_counts, step, lam=MMR_LAMBDA):
    return lam * rel_value - (1 - lam) * mmr_max_sim(cand_idx, selected)

def mmr_rerank(cand_ids, cand_rel, lam=MMR_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, mmr_score, k=k, lam=lam)

# ── Strategy D: Fairness-only re-ranking, no diversity term ──────────────
def fairness_only_score(cand_idx, rel_value, selected, tier_counts, step, lam_f=FAIR_ONLY_LAMBDA):
    return (1 - lam_f) * rel_value + lam_f * fair_boost(cand_idx, tier_counts, step)

def fairness_only_rerank(cand_ids, cand_rel, lam_f=FAIR_ONLY_LAMBDA, k=K):
    return greedy_rerank(cand_ids, cand_rel, fairness_only_score, k=k, lam_f=lam_f)

print('Strategy C (MMR, diversity-only) defined ✓  —  score = λ·rel − (1−λ)·max_sim,  λ =', MMR_LAMBDA)
print('Strategy D (fairness-only) defined ✓        —  score = (1−λ_f)·rel + λ_f·fair_boost,  λ_f =', FAIR_ONLY_LAMBDA)


---
## 4b. Lambda Grid Search (Bayesian Optimisation via Optuna)

In [ ]:
# Lambda grid-search hyper-params
LAMBDA_STEP  = 0.10
VAL_FRACTION = 0.10
BETA_F       = 0.5
BETA_D       = 0.5
# MMR_LAMBDA / FAIR_ONLY_LAMBDA are defined earlier, in the baseline-reranker cell (Section 4a), since they're needed there first.
print('Lambda grid-search config loaded.')


### Core Metric Helpers

`ild`, `precision_recall_ndcg`, and `novelty_fairness` are defined once, here,
because they're needed both inside the Optuna objective below (Section 4b)
and later throughout evaluation (Section 5 onward) — defining them twice was
redundant and risked the two copies drifting apart.


In [ ]:
# ── Core metric helpers (single canonical definition, reused everywhere) ─────
def ild(rec_list):
    """Intra-list diversity: 1 - mean pairwise genre-cosine similarity."""
    if len(rec_list) < 2:
        return 0.0
    idx = np.array(rec_list)
    sub = GENRE_SIM[np.ix_(idx, idx)]   # ⚡ vectorised: submatrix of pre-computed GENRE_SIM
    n = len(idx)
    mask = np.triu(np.ones((n, n), dtype=bool), k=1)
    return float(1.0 - sub[mask].mean())

def precision_recall_ndcg(rec_list, relevant_set, grades, k=K):
    rec   = rec_list[:k]
    hits  = len(set(rec) & relevant_set)
    prec  = hits / k
    rec_r = hits / max(1, len(relevant_set))
    dcg   = sum(grades.get(item, 0) / np.log2(r + 2) for r, item in enumerate(rec))
    idcg  = sum(g / np.log2(r + 2) for r, g in enumerate(sorted(grades.values(), reverse=True)[:k]))
    return prec, rec_r, (dcg / idcg if idcg > 0 else 0.0)

def novelty_fairness(rec_list):
    """Mean (1 - normalised popularity): higher = more novel/niche items recommended."""
    return float(np.mean([1 - pop_norm[i] for i in rec_list]))

print('Core metric helpers (ild, precision_recall_ndcg, novelty_fairness) defined ✓')


In [ ]:
# Step 1: carve validation split from training data
train_df_sorted = train_df.sort_values(['UserID','Timestamp']).reset_index(drop=True)
train_df_sorted['rank_desc_tr'] = train_df_sorted.groupby('UserID').cumcount(ascending=False)
train_df_sorted['user_count_tr'] = train_df_sorted.groupby('UserID')['UserID'].transform('count')
val_cutoff   = np.maximum(1, (train_df_sorted['user_count_tr'] * VAL_FRACTION).astype(int))
val_mask_tr  = train_df_sorted['rank_desc_tr'] < val_cutoff

val_tr_df   = train_df_sorted.loc[ val_mask_tr].copy()
pure_tr_df  = train_df_sorted.loc[~val_mask_tr].copy()

val_relevant_tr = val_tr_df[val_tr_df['Rating'] >= 4].groupby('u_idx')['i_idx'].apply(set).to_dict()
val_grades_tr   = val_tr_df.groupby('u_idx').apply(lambda g: dict(zip(g['i_idx'], g['Rating']))).to_dict()
pure_seen_tr    = pure_tr_df.groupby('u_idx')['i_idx'].apply(set).to_dict()

# Validation pools also use the fast matrix approach
def build_val_score_matrix_svd(model, pure_seen_dict):
    """Same matrix, but mask pure_seen items per user before argpartition."""
    mat = build_score_matrix_svd(model).copy()
    for u, seen in pure_seen_dict.items():
        mat[u, list(seen)] = -np.inf
    return mat

def build_val_score_matrix_bpr(model, pure_seen_dict):
    mat = build_score_matrix_bpr(model).copy()
    for u, seen in pure_seen_dict.items():
        mat[u, list(seen)] = -np.inf
    return mat

print('Building validation candidate pools ...')
svd_val_mat  = build_val_score_matrix_svd(svd_model, pure_seen_tr)
bpr_val_mat  = build_val_score_matrix_bpr(bpr_model, pure_seen_tr)
svd_val_pools = build_candidates_fast(svd_val_mat, {})   # seen already masked in matrix
bpr_val_pools = build_candidates_fast(bpr_val_mat, {})
print(f'  SVD val pools: {len(svd_val_pools)}   BPR val pools: {len(bpr_val_pools)}')

# Step 2: diversity target = mean ILD of Top-K on val (uses the canonical `ild` defined above)
def mean_ild_topk(val_pools):
    return float(np.mean([ild(topk_rerank(cids, crel)) for _, (cids, crel) in val_pools.items()]))

TARGET_D_SVD = mean_ild_topk(svd_val_pools)
TARGET_D_BPR = mean_ild_topk(bpr_val_pools)
print(f'Diversity targets — SVD: {TARGET_D_SVD:.4f}   BPR: {TARGET_D_BPR:.4f}')


In [ ]:


# Composite objective for the Optuna search (uses the canonical metric helpers defined above)
def composite_score_user(rec, relevant_set, grades, target_d):
    if not rec:
        return -np.inf
    _, _, ndcg = precision_recall_ndcg(rec, relevant_set, grades)
    d_val = ild(rec)
    f_val = novelty_fairness(rec)
    return ndcg - BETA_F * abs(f_val - 1/3) - BETA_D * abs(d_val - target_d)

def optimize_lambdas(val_pools, target_d, name, n_trials=50):
    users = [(u, cids, crel) for u, (cids, crel) in val_pools.items()]
    history = []

    def objective(trial):
        # Restrict search to a clean 2-decimal-place grid: 0.00, 0.01, ..., 1.00
        lam_f = round(trial.suggest_float("lam_f", 0.0, 1.0, step=0.01), 2)

        hi = round(1.0 - lam_f, 2)
        if hi <= 0:
            lam_d = 0.0
            trial.set_user_attr("lam_d_forced", True)
        else:
            lam_d = round(trial.suggest_float("lam_d", 0.0, hi, step=0.01), 2)

        scores = []
        for u, cids, crel in users:
            rel = val_relevant_tr.get(u, set())
            grd = val_grades_tr.get(u, {})
            rec = jfds_rerank(cids, crel, lam_f=lam_f, lam_d=lam_d)
            scores.append(composite_score_user(rec, rel, grd, target_d))

        mean_score = float(np.mean(scores))
        history.append({'lam_f': lam_f, 'lam_d': lam_d, 'composite': mean_score})
        return mean_score

    study = optuna.create_study(direction="maximize",
                                 sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best_lf = round(study.best_params["lam_f"], 2)
    best_ld = round(study.best_params.get("lam_d", 0.0), 2)
    best_lu = round(1 - best_lf - best_ld, 2)

    print(f"[{name}] best λ_f={best_lf:.2f} λ_d={best_ld:.2f} "
          f"λ_u={best_lu:.2f} composite={study.best_value:.5f}")

    return best_lf, best_ld, pd.DataFrame(history)

print("Running Bayesian Optimisation ...")
best_lf_svd, best_ld_svd, optuna_svd = optimize_lambdas(svd_val_pools, TARGET_D_SVD, "SVD", n_trials=50)
best_lf_bpr, best_ld_bpr, optuna_bpr = optimize_lambdas(bpr_val_pools, TARGET_D_BPR, "BPR", n_trials=50)

best_lambdas = {
    "SVD": {"lam_f": best_lf_svd, "lam_d": best_ld_svd},
    "BPR": {"lam_f": best_lf_bpr, "lam_d": best_ld_bpr},
}
print("\nBest lambdas:", best_lambdas)

## Hyperparameter Optimization

The optimal fairness (`λ_f`) and diversity (`λ_d`) weights are selected using a validation set and Bayesian Optimization.

- Split the training data into training and validation subsets.
- Generate validation candidate pools for the SVD and BPR models.
- Compute a target diversity score from the validation recommendations.
- Evaluate recommendations using a composite objective that balances ranking quality (NDCG), fairness, and diversity.
- Use Bayesian Optimization (Optuna) to find the best values of `λ_f` and `λ_d` for each model.

In [ ]:
# Visualise Optuna trial history as scatter (lam_f vs lam_d, colour = composite)
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, (name, hist_df), best_lf, best_ld in zip(
        axes,
        [('SVD', optuna_svd), ('BPR', optuna_bpr)],
        [best_lf_svd, best_lf_bpr],
        [best_ld_svd, best_ld_bpr]):
    sc = ax.scatter(hist_df['lam_f'], hist_df['lam_d'],
                    c=hist_df['composite'], cmap='RdYlGn', s=60, edgecolor='k', linewidth=0.4)
    ax.scatter([best_lf], [best_ld], marker='*', s=280, color='white',
               edgecolor='black', linewidth=1.2, zorder=5,
               label=f'best: λ_f={best_lf:.2f}, λ_d={best_ld:.2f}')
    ax.set_xlabel('λ_f (fairness weight)')
    ax.set_ylabel('λ_d (diversity weight)')
    ax.set_title(f'{name}: Optuna trial history\n(white star = selected λ)', fontsize=11)
    ax.legend(fontsize=9)
    plt.colorbar(sc, ax=ax, label='composite score')

plt.suptitle('Lambda Search — Bayesian Optimisation', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('lambda_grid_search.png', dpi=110, bbox_inches='tight')
plt.show()


In [ ]:
# Re-rank all users using best lambdas (JFDS) plus the two fixed-weight baselines (MMR, Fairness-only)
print('Re-ranking all users (Top-K, JFDS, MMR, Fairness-only) ...')
lists = {}
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    lf = best_lambdas[model_name]['lam_f']
    ld = best_lambdas[model_name]['lam_d']
    topk_lists, jfds_lists, mmr_lists, fair_lists = {}, {}, {}, {}
    for u in range(N_USERS):
        cand_ids, cand_rel = pools[u]
        if len(cand_ids) == 0:
            continue
        topk_lists[u] = topk_rerank(cand_ids, cand_rel)
        jfds_lists[u] = jfds_rerank(cand_ids, cand_rel, lam_f=lf, lam_d=ld)
        mmr_lists[u]  = mmr_rerank(cand_ids, cand_rel)
        fair_lists[u] = fairness_only_rerank(cand_ids, cand_rel)
    lists[(model_name, 'TopK')]    = topk_lists
    lists[(model_name, 'JFDS')]    = jfds_lists
    lists[(model_name, 'MMR')]     = mmr_lists
    lists[(model_name, 'FairOnly')] = fair_lists
    print(f"  {model_name}: λ_f={lf:.2f} λ_d={ld:.2f}  |  MMR λ={MMR_LAMBDA}  |  FairOnly λ_f={FAIR_ONLY_LAMBDA}  — "
          f"{len(topk_lists)} Top-K, {len(jfds_lists)} JFDS, {len(mmr_lists)} MMR, {len(fair_lists)} FairOnly lists")


---
## Sanity Check: One User, Four Strategies Side by Side

Before trusting the aggregate metrics below, spot-check a single user: print
the Top-10 list each strategy actually produces for them (SVD candidates),
so the numbers in the rest of the notebook can be tied back to real,
readable recommendations.


In [ ]:
# ── Single-user sanity check (SVD candidates) ──────────────────────
SAMPLE_U = next(iter(lists[('SVD', 'JFDS')].keys()))
idx2movie = {v: k for k, v in movie2idx.items()}

sample_rows = []
for method in ['TopK', 'JFDS', 'MMR', 'FairOnly']:
    rec = lists[('SVD', method)].get(SAMPLE_U, [])
    titles = movies_df.set_index('MovieID').loc[[idx2movie[i] for i in rec], 'Title'].tolist()
    for rank, (item_idx, title) in enumerate(zip(rec, titles), start=1):
        sample_rows.append({'Strategy': method, 'Rank': rank, 'Title': title, 'Tier': tier[item_idx]})

sample_df = pd.DataFrame(sample_rows)
print(f"Sample user (SVD, u_idx={SAMPLE_U}) — already rated {len(train_seen.get(SAMPLE_U, set()))} movies in training")
for method in ['TopK', 'JFDS', 'MMR', 'FairOnly']:
    print(f"\n=== {method} ===")
    print(sample_df[sample_df.Strategy == method][['Rank', 'Title', 'Tier']].to_string(index=False))

ids_topk = set(lists[('SVD', 'TopK')].get(SAMPLE_U, []))
ids_jfds = set(lists[('SVD', 'JFDS')].get(SAMPLE_U, []))
overlap  = ids_topk & ids_jfds
print(f"\nTop-K vs JFDS overlap for this user: {len(overlap)} / {len(ids_topk)} items in common")


---
## 5. Evaluation Metrics 

In [ ]:
# `precision_recall_ndcg`, `ild`, and `novelty_fairness` are already defined above (Section 4b) and reused here unchanged.
def shannon_entropy(rec_list):
    dist = genre_vec[rec_list].mean(axis=0)
    dist = dist / dist.sum()
    return float(-np.sum([p * np.log2(p) for p in dist if p > 0]))

def gini(values):
    v = np.sort(np.asarray(values, dtype=float))
    n = len(v)
    if v.sum() == 0:
        return 0.0
    cum = np.cumsum(v)
    return float((n + 1 - 2 * np.sum(cum) / cum[-1]) / n)

def calibration_kl(rec_list, user_train_items, alpha=0.01):
    if len(user_train_items) == 0:
        return 0.0
    p = genre_vec[list(user_train_items)].mean(axis=0)
    p = p / p.sum()
    q = genre_vec[rec_list].mean(axis=0)
    q = (1 - alpha) * q + alpha * p
    q = q / q.sum()
    return float(np.sum(p * np.log((p + 1e-12) / (q + 1e-12))))

# ── Aggregate diversity helpers (for R10) ─────────────────────────────────────
def aggregate_diversity(rec_lists_dict):
    all_items = set()
    for rec in rec_lists_dict.values():
        all_items.update(rec)
    return len(all_items)

def exposure_vector(rec_lists_dict, n_items=N_ITEMS):
    exp = np.zeros(n_items)
    for rec in rec_lists_dict.values():
        for i in rec:
            exp[i] += 1
    return exp


In [ ]:
# Per-user metrics for every (base_model, method) combination
rows = []
for (model_name, method), rec_lists in lists.items():
    for u, rec in rec_lists.items():
        relevant_set = test_relevant.get(u, set())
        grades       = test_grades.get(u, {})
        p, r, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
        rows.append({
            'user': u, 'base_model': model_name, 'method': method,
            'precision': p, 'recall': r, 'ndcg': n_ndcg,
            'D': ild(rec), 'F': novelty_fairness(rec),
            'H': shannon_entropy(rec), 'gini_exp': gini(pop_count[rec]),
            'calibration_kl': calibration_kl(rec, train_seen.get(u, set())),
        })

metrics_long = pd.DataFrame(rows)
metrics_long['U'] = metrics_long['ndcg']
print(f"metrics_long: {len(metrics_long):,} rows")
metrics_long.head()


In [ ]:
# Headline comparison table
summary = (metrics_long
           .groupby(['base_model', 'method'])
           [['precision', 'recall', 'ndcg', 'D', 'F', 'H', 'gini_exp', 'calibration_kl']]
           .mean().round(4))

sys_rows = []
for (model_name, method), rec_lists in lists.items():
    exposure   = exposure_vector(rec_lists)
    coverage   = (exposure > 0).sum() / N_ITEMS
    exp_gini   = gini(exposure)
    agg_div    = aggregate_diversity(rec_lists)
    tier_share = (pd.Series(tier[np.repeat(np.arange(N_ITEMS), exposure.astype(int))])
                  .value_counts(normalize=True).reindex(TIERS).fillna(0))
    sys_rows.append({'base_model': model_name, 'method': method,
                     'agg_diversity': agg_div, 'coverage': coverage,
                     'exposure_gini': exp_gini,
                     'head_share': tier_share['head'],
                     'mid_share':  tier_share['mid'],
                     'tail_share': tier_share['tail']})

system_summary = pd.DataFrame(sys_rows).set_index(['base_model', 'method']).round(4)
display(summary)
display(system_summary)


---
## Baseline Comparison — JFDS vs Top-K vs MMR vs Fairness-only

The headline table above already contains all four methods (Top-K, JFDS,
MMR, Fairness-only) for both base recommenders, because they were folded
into the same `lists` / `metrics_long` structures used everywhere else in
this notebook. This section just visualizes that comparison directly:

1. Bar charts of Fairness (F), Diversity (D), and NDCG for all four
   strategies, per base model.
2. A delta table quantifying exactly what JFDS gains or gives up relative
   to each specialist baseline.


In [ ]:
# ── Bar chart: F / D / NDCG across all 4 strategies, per base model ────────
method_order = ['TopK', 'MMR', 'FairOnly', 'JFDS']
method_colors = {'TopK': '#7F8C8D', 'MMR': '#2980B9', 'FairOnly': '#E67E22', 'JFDS': '#27AE60'}
metrics_bar = ['F', 'D', 'ndcg']
metric_titles = {'F': 'Fairness (novelty)', 'D': 'Diversity (ILD)', 'ndcg': 'NDCG@10 (utility)'}

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for row, model_name in enumerate(['SVD', 'BPR']):
    for col, metric in enumerate(metrics_bar):
        ax = axes[row][col]
        vals = [metrics_long[(metrics_long.base_model == model_name) &
                              (metrics_long.method == m)][metric].mean() for m in method_order]
        bars = ax.bar(method_order, vals, color=[method_colors[m] for m in method_order], edgecolor='black')
        for b, v in zip(bars, vals):
            ax.text(b.get_x() + b.get_width()/2, v, f'{v:.3f}', ha='center', va='bottom', fontsize=8)
        ax.set_title(f'{model_name}: {metric_titles[metric]}', fontsize=10)
        ax.tick_params(axis='x', rotation=20)

plt.suptitle('JFDS vs. Top-K, MMR (diversity-only), and Fairness-only baselines', y=1.02, fontsize=13)
plt.tight_layout()
plt.savefig('baseline_comparison_bars.png', dpi=110, bbox_inches='tight')
plt.show()

# ── Delta table: JFDS vs each established baseline, per base model ──────────
delta_rows_base = []
for model_name in ['SVD', 'BPR']:
    jfds_means = metrics_long[(metrics_long.base_model == model_name) &
                               (metrics_long.method == 'JFDS')][metrics_bar].mean()
    for base_method in ['TopK', 'MMR', 'FairOnly']:
        base_means = metrics_long[(metrics_long.base_model == model_name) &
                                   (metrics_long.method == base_method)][metrics_bar].mean()
        for metric in metrics_bar:
            b_val, base_val = jfds_means[metric], base_means[metric]
            pct = (b_val - base_val) / base_val * 100 if base_val != 0 else np.nan
            delta_rows_base.append({
                'base_model': model_name, 'baseline': base_method, 'metric': metric,
                'JFDS': round(b_val, 4), 'baseline_value': round(base_val, 4),
                'delta': round(b_val - base_val, 4), 'pct_change': round(pct, 2),
            })

baseline_delta_df = pd.DataFrame(delta_rows_base)
display(baseline_delta_df)


---
## R1 — Pareto Frontier (SVD vs. BPR)

In [ ]:
def pareto_mask(vals):
    n = len(vals)
    dominated = np.zeros(n, dtype=bool)
    for i in range(n):
        if dominated[i]:
            continue
        diff = vals - vals[i]
        better_eq = np.all(diff >= -1e-12, axis=1)
        strictly  = np.any(diff >  1e-9,  axis=1)
        dom = better_eq & strictly
        dom[i] = False
        if dom.any():
            dominated[i] = True
    return ~dominated

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
pareto_stats = {}

for ax, model_name, color in zip(axes, ['SVD', 'BPR'], [C_SVD, C_BPR]):
    jfds_pts = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')]
    topk_pts = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'TopK')]
    jfds_s = jfds_pts.sample(min(ANALYSIS_SAMPLE_N, len(jfds_pts)), random_state=RANDOM_SEED)
    topk_s = topk_pts.sample(min(ANALYSIS_SAMPLE_N, len(topk_pts)), random_state=RANDOM_SEED)

    vals = jfds_s[['F', 'D', 'U']].values
    mask = pareto_mask(vals)

    ax.scatter(topk_s['F'], topk_s['D'], s=14, c='lightgrey', alpha=0.5, label='Top-K (reference)')
    ax.scatter(jfds_s.loc[~mask, 'F'], jfds_s.loc[~mask, 'D'], s=16, c=color, alpha=0.35, label='JFDS (dominated)')
    ax.scatter(jfds_s.loc[ mask, 'F'], jfds_s.loc[ mask, 'D'], s=34, c=color,
               edgecolor='black', linewidth=0.5, label='JFDS (Pareto-optimal)')
    ax.set_xlabel('Fairness (F)'); ax.set_ylabel('Diversity (D)')
    ax.set_title(f'{model_name}: F-D Pareto Frontier', fontsize=11)
    ax.legend(fontsize=8, loc='lower right')

    r_pareto = np.corrcoef(jfds_s.loc[mask, 'F'], jfds_s.loc[mask, 'D'])[0, 1] if mask.sum() > 2 else np.nan
    topk_vals = topk_s[['F', 'D', 'U']].values
    topk_dominated = sum(
        1 for i in range(len(topk_s))
        if (np.all(vals - topk_vals[i] >= -1e-12, axis=1) & np.any(vals - topk_vals[i] > 1e-9, axis=1)).any()
    )
    pareto_stats[model_name] = {
        'n_pareto_optimal_jfds': int(mask.sum()),
        'n_jfds_sampled': len(jfds_s),
        'pareto_pearson_r': r_pareto,
        'pct_topk_dominated_by_jfds': 100 * topk_dominated / len(topk_s),
    }

plt.tight_layout()
plt.savefig('r1_pareto_frontier.png', dpi=110, bbox_inches='tight')
plt.show()
pd.DataFrame(pareto_stats).T.round(3)


---
## R2 — Scaling Law (AIC/BIC)

In [ ]:
jfds_all = metrics_long[metrics_long.method == 'JFDS']
pool = jfds_all.sample(min(ANALYSIS_SAMPLE_N * 2, len(jfds_all)), random_state=RANDOM_SEED)
F_vals = pool['F'].values
D_vals = pool['D'].values
eps = 1e-6

def aic_bic(y, y_pred, k_params):
    n   = len(y)
    rss = max(np.sum((y - y_pred) ** 2), 1e-12)
    return n * np.log(rss/n) + 2 * k_params, n * np.log(rss/n) + k_params * np.log(n)

models_fit = {}
c1 = np.polyfit(F_vals, D_vals, 1)
models_fit['Linear'] = {'params': c1, 'pred': np.polyval(c1, F_vals), 'k': 2,
                        'eq': f"D = {c1[1]:.4f} + {c1[0]:.4f}*F"}
c2 = np.polyfit(F_vals, D_vals, 2)
models_fit['Quadratic'] = {'params': c2, 'pred': np.polyval(c2, F_vals), 'k': 3,
                           'eq': f"D = {c2[2]:.4f} + {c2[1]:.4f}*F + {c2[0]:.4f}*F^2"}

for name, fn, p0 in [
    ('Power',      lambda x, a, b: a * np.power(x + eps, b),          [1, 0.5]),
    ('Log',        lambda x, a, b: a + b * np.log(x + eps),            [0, 0.1]),
    ('Saturation', lambda x, a, b: a * x / (b + x + eps),              [1, 0.1]),
]:
    try:
        popt, _ = curve_fit(fn, F_vals, D_vals, p0=p0, maxfev=10000)
        models_fit[name] = {'params': popt, 'pred': fn(F_vals, *popt), 'k': 2,
                            'eq': f"{name} fit (a={popt[0]:.4f}, b={popt[1]:.4f})"}
    except RuntimeError:
        pass

rows_r2 = []
for name, m in models_fit.items():
    aic, bic = aic_bic(D_vals, m['pred'], m['k'])
    rows_r2.append({'Model': name, 'Equation': m['eq'], 'AIC': aic, 'BIC': bic})
scaling_table = pd.DataFrame(rows_r2).sort_values('AIC').reset_index(drop=True)
display(scaling_table)

best_name = scaling_table.iloc[0]['Model']
print(f"Best model by AIC: {best_name}  ->  {scaling_table.iloc[0]['Equation']}")

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(F_vals, D_vals, s=8, alpha=0.25, color='grey', label='data (JFDS lists)')
F_grid = np.linspace(F_vals.min(), F_vals.max(), 200)
for (name, m), col in zip(models_fit.items(), plt.cm.tab10(np.linspace(0, 1, len(models_fit)))):
    try:
        if name in ('Linear', 'Quadratic'):
            yg = np.polyval(m['params'], F_grid)
        else:
            from scipy.optimize import curve_fit as _cf
            fn_map = {
                'Power':      lambda x, a, b: a * np.power(x + eps, b),
                'Log':        lambda x, a, b: a + b * np.log(x + eps),
                'Saturation': lambda x, a, b: a * x / (b + x + eps),
            }
            yg = fn_map[name](F_grid, *m['params'])
        lw = 2.5 if name == best_name else 1.2
        ax.plot(F_grid, yg, color=col, linewidth=lw,
                label=f"{name}{' (best)' if name == best_name else ''}")
    except Exception:
        pass
ax.set_xlabel('Fairness (F)'); ax.set_ylabel('Diversity (D)')
ax.set_title('R2 — Scaling Law: D as a function of F', fontsize=12)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('r2_scaling_law.png', dpi=110, bbox_inches='tight')
plt.show()


---
## R3 — Regime Analysis

In [ ]:
pool = pool.copy()
pool['F_quintile'] = pd.qcut(pool['F'], 5, labels=[f'Q{i+1}' for i in range(5)], duplicates='drop')

slope_rows = []
for q, g in pool.groupby('F_quintile', observed=True):
    if len(g) < 3 or g['F'].std() < 1e-9:
        continue
    c, cov = np.polyfit(g['F'], g['D'], 1, cov=True)
    slope_rows.append({'quintile': q, 'F_range': f"[{g['F'].min():.3f}, {g['F'].max():.3f}]",
                       'n': len(g), 'slope': c[0], 'slope_se': np.sqrt(cov[0, 0])})

regime_df = pd.DataFrame(slope_rows)
display(regime_df.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(regime_df['quintile'].astype(str), regime_df['slope'],
       yerr=regime_df['slope_se'] * 1.96,
       color=[C_JFDS if s > 0 else C_TOPK for s in regime_df['slope']], capsize=4)
ax.axhline(0, color='black', lw=1)
ax.set_xlabel('Fairness (F) quintile')
ax.set_ylabel('Local OLS slope  dD/dF  (±95% CI)')
ax.set_title('R3 — Regime Analysis', fontsize=12)
plt.tight_layout()
plt.savefig('r3_regime_analysis.png', dpi=110, bbox_inches='tight')
plt.show()

n_pos = int((regime_df['slope'] > 0).sum())
n_neg = int((regime_df['slope'] < 0).sum())
print(f"Positive-slope regimes: {n_pos}/{len(regime_df)}   Negative: {n_neg}/{len(regime_df)}")


---
## R4 — Conservation Invariant

In [ ]:
F_arr = pool['F'].values + 1e-6
D_arr = pool['D'].values + 1e-6
alphas = np.round(np.arange(-2, 2.01, 0.2), 2)
betas  = np.round(np.arange(-2, 2.01, 0.2), 2)

def cv_grid(F_arr, D_arr, alphas, betas):
    grid = np.zeros((len(alphas), len(betas)))
    for ai, a in enumerate(alphas):
        Fp = F_arr ** a
        for bi, b in enumerate(betas):
            Q  = Fp * (D_arr ** b)
            m  = np.mean(Q)
            grid[ai, bi] = np.std(Q) / abs(m) if abs(m) > 1e-9 else np.inf
    return grid

grid = cv_grid(F_arr, D_arr, alphas, betas)
best_idx = np.unravel_index(np.argmin(grid), grid.shape)
best_alpha, best_beta = alphas[best_idx[0]], betas[best_idx[1]]
best_cv = grid[best_idx]
print(f"Best invariant: F^{best_alpha} * D^{best_beta}   (CV = {best_cv:.4f})")

B, n = 200, len(F_arr)
boot_alpha, boot_beta = [], []
for _ in range(B):
    idx = RNG.integers(0, n, size=n)
    g   = cv_grid(F_arr[idx], D_arr[idx], alphas, betas)
    bi  = np.unravel_index(np.argmin(g), g.shape)
    boot_alpha.append(alphas[bi[0]]); boot_beta.append(betas[bi[1]])

alpha_ci = np.percentile(boot_alpha, [2.5, 97.5])
beta_ci  = np.percentile(boot_beta,  [2.5, 97.5])
print(f"Bootstrap 95% CI:  alpha [{alpha_ci[0]:.2f}, {alpha_ci[1]:.2f}]   "
      f"beta [{beta_ci[0]:.2f}, {beta_ci[1]:.2f}]")

fig, ax = plt.subplots(figsize=(7.5, 6))
im = ax.imshow(grid.T, origin='lower',
               extent=[alphas.min(), alphas.max(), betas.min(), betas.max()],
               aspect='auto', cmap='viridis_r')
ax.scatter([best_alpha], [best_beta], color='red', marker='*', s=200,
           label=f'best (CV={best_cv:.3f})')
ax.set_xlabel(r'$\alpha$'); ax.set_ylabel(r'$\beta$')
ax.set_title(r'R4 — CV of $F^\alpha D^\beta$', fontsize=12)
plt.colorbar(im, ax=ax, label='CV (lower = more conserved)')
ax.legend()
plt.tight_layout()
plt.savefig('r4_conservation_invariant.png', dpi=110, bbox_inches='tight')
plt.show()


---
## R5 — Coupling Coefficient ρ_FD

In [ ]:
order    = np.argsort(pool['F'].values)
F_sorted = pool['F'].values[order]
D_sorted = pool['D'].values[order]
U_sorted = pool['U'].values[order]
n, half  = len(F_sorted), 25

rho_fd = np.full(n, np.nan)
for pos in range(n):
    lo, hi = max(0, pos - half), min(n, pos + half + 1)
    Fw, Dw = F_sorted[lo:hi], D_sorted[lo:hi]
    if len(Fw) > 5 and np.var(Fw) > 1e-9:
        rho_fd[pos] = np.cov(Fw, Dw)[0, 1] / np.var(Fw)

valid = ~np.isnan(rho_fd)
r_coupling = np.corrcoef(rho_fd[valid], U_sorted[valid])[0, 1]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.scatter(U_sorted[valid], rho_fd[valid], s=10, alpha=0.35, color=C_JFDS)
c  = np.polyfit(U_sorted[valid], rho_fd[valid], 1)
xg = np.linspace(U_sorted[valid].min(), U_sorted[valid].max(), 100)
ax.plot(xg, np.polyval(c, xg), color='black', lw=2,
        label=f'trend (Pearson r={r_coupling:.3f})')
ax.axhline(0, color='grey', ls='--', lw=1)
ax.set_xlabel('Utility (U = NDCG@10)')
ax.set_ylabel(r'Local coupling $\rho_{{FD}}$ (50-NN slope)')
ax.set_title('R5 — Coupling Coefficient vs. Utility', fontsize=12)
ax.legend()
plt.tight_layout()
plt.savefig('r5_coupling_coefficient.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"Pearson r(rho_FD, Utility) = {r_coupling:.4f}")


---
## R6 — Information Theory

In [ ]:
H_vals        = pool['H'].values
one_minus_gini = 1 - pool['gini_exp'].values

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))

c1    = np.polyfit(H_vals, pool['D'].values, 1)
pred1 = np.polyval(c1, H_vals)
r2_1  = 1 - np.sum((pool['D'].values - pred1)**2) / np.sum((pool['D'].values - pool['D'].values.mean())**2)
axes[0].scatter(H_vals, pool['D'], s=10, alpha=0.3, color=C_SVD)
xg = np.linspace(H_vals.min(), H_vals.max(), 100)
axes[0].plot(xg, np.polyval(c1, xg), color='black', lw=2, label=f'R²={r2_1:.3f}')
axes[0].set_xlabel('Shannon entropy H'); axes[0].set_ylabel('Diversity (ILD)')
axes[0].set_title('Is D proportional to entropy?', fontsize=11)
axes[0].legend()

c2    = np.polyfit(one_minus_gini, pool['F'].values, 1)
pred2 = np.polyval(c2, one_minus_gini)
r2_2  = 1 - np.sum((pool['F'].values - pred2)**2) / np.sum((pool['F'].values - pool['F'].values.mean())**2)
axes[1].scatter(one_minus_gini, pool['F'], s=10, alpha=0.3, color=C_BPR)
xg2 = np.linspace(one_minus_gini.min(), one_minus_gini.max(), 100)
axes[1].plot(xg2, np.polyval(c2, xg2), color='black', lw=2, label=f'R²={r2_2:.3f}')
axes[1].set_xlabel('1 − Gini(popularity)'); axes[1].set_ylabel('Fairness (novelty)')
axes[1].set_title('Is F proportional to 1 − Gini?', fontsize=11)
axes[1].legend()

plt.suptitle('R6 — Information-Theoretic Checks', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('r6_information_theory.png', dpi=110, bbox_inches='tight')
plt.show()
print(f"D ~ H:          R² = {r2_1:.4f}")
print(f"F ~ 1-Gini:     R² = {r2_2:.4f}")


---
## R7 — Paired Significance Tests

In [ ]:
sig_rows = []
BASELINE_METHODS = ['TopK', 'MMR', 'FairOnly']

for model_name in ['SVD', 'BPR']:
    jfds_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')].set_index('user')

    for baseline_method in BASELINE_METHODS:
        base_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == baseline_method)].set_index('user')
        common  = base_df.index.intersection(jfds_df.index)
        base_c, jfds_c = base_df.loc[common], jfds_df.loc[common]

        for metric in ['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']:
            a, b = base_c[metric].values, jfds_c[metric].values
            diff = b - a
            try:
                w_stat, w_p = stats.wilcoxon(a, b)
            except ValueError:
                w_stat, w_p = np.nan, np.nan
            t_stat, t_p = stats.ttest_rel(b, a)
            sig_rows.append({
                'base_model': model_name, 'baseline': baseline_method, 'metric': metric,
                'mean_baseline': a.mean(), 'mean_JFDS': b.mean(),
                'mean_delta': diff.mean(), 'wilcoxon_p': w_p, 'ttest_p': t_p,
                'JFDS_wins': int(np.sum(diff > 1e-9)),
                'JFDS_losses': int(np.sum(diff < -1e-9)),
                'ties': len(diff) - int(np.sum(diff > 1e-9)) - int(np.sum(diff < -1e-9)),
                'n_users': len(common),
            })

sig_table = pd.DataFrame(sig_rows).round(5)
sig_table


---
## R7b — Holm–Bonferroni Correction

`sig_table` above now holds `36` separate per-user paired tests (3 baselines
× 6 metrics × 2 base models). Reporting raw p-values from that many
comparisons without correction overstates significance — a Holm–Bonferroni
correction controls the family-wise error rate across all of them at once.


In [ ]:
# ── Holm–Bonferroni correction across all per-user paired tests ──────────
try:
    from statsmodels.stats.multitest import multipletests
    HAVE_STATSMODELS = True
except ImportError:
    HAVE_STATSMODELS = False
    print("statsmodels not available — falling back to a manual Holm–Bonferroni implementation.")

def holm_bonferroni(pvals, alpha=0.05):
    """Manual fallback: returns (reject_bool_array, corrected_pvals) in original order."""
    pvals = np.asarray(pvals, dtype=float)
    n = len(pvals)
    order = np.argsort(pvals)
    corrected = np.empty(n)
    running_max = 0.0
    for rank, i in enumerate(order):
        adj = (n - rank) * pvals[i]
        running_max = max(running_max, adj)
        corrected[i] = min(running_max, 1.0)
    reject = corrected < alpha
    return reject, corrected

ALPHA = 0.05
pvals = sig_table['wilcoxon_p'].fillna(1.0).values

if HAVE_STATSMODELS:
    reject, p_corrected, _, _ = multipletests(pvals, alpha=ALPHA, method='holm')
else:
    reject, p_corrected = holm_bonferroni(pvals, alpha=ALPHA)

sig_table['wilcoxon_p_holm'] = p_corrected
sig_table['significant_holm'] = reject

print('=== Per-user paired tests: JFDS vs each baseline, AFTER Holm–Bonferroni correction ===')
display_cols = ['base_model', 'baseline', 'metric', 'mean_delta', 'wilcoxon_p',
                 'wilcoxon_p_holm', 'significant_holm']
print(sig_table[display_cols].to_string(index=False))

n_sig_holm = int(sig_table['significant_holm'].sum())
print(f"\n{n_sig_holm} / {len(sig_table)} comparisons remain significant after Holm correction (α={ALPHA}).")


---
## R8 — Bootstrap Confidence Intervals

In [ ]:
def bootstrap_delta_ci(a, b, n_boot=2000, seed=RANDOM_SEED):
    rng   = np.random.default_rng(seed)
    diffs = b - a
    boots = np.array([rng.choice(diffs, size=len(diffs), replace=True).mean() for _ in range(n_boot)])
    return diffs.mean(), np.percentile(boots, [2.5, 97.5])

boot_rows = []
for model_name in ['SVD', 'BPR']:
    jfds_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == 'JFDS')].set_index('user')
    for baseline_method in BASELINE_METHODS:
        base_df = metrics_long[(metrics_long.base_model == model_name) & (metrics_long.method == baseline_method)].set_index('user')
        common  = base_df.index.intersection(jfds_df.index)
        base_c, jfds_c = base_df.loc[common], jfds_df.loc[common]
        for metric in ['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']:
            mean_delta, ci = bootstrap_delta_ci(base_c[metric].values, jfds_c[metric].values)
            boot_rows.append({'base_model': model_name, 'baseline': baseline_method, 'metric': metric,
                              'mean_delta': mean_delta, 'ci_low': ci[0], 'ci_high': ci[1]})

boot_df = pd.DataFrame(boot_rows)
display(boot_df.round(5))

fig, axes = plt.subplots(len(BASELINE_METHODS), 2, figsize=(13, 14), sharey=False)
for row, baseline_method in enumerate(BASELINE_METHODS):
    for col, model_name in enumerate(['SVD', 'BPR']):
        ax  = axes[row][col]
        sub = boot_df[(boot_df.base_model == model_name) & (boot_df.baseline == baseline_method)].reset_index(drop=True)
        y   = np.arange(len(sub))
        colors_b = [C_JFDS if (lo > 0 or hi < 0) else 'grey' for lo, hi in zip(sub.ci_low, sub.ci_high)]

        # ecolor doesn't accept a per-point color array, so draw each errorbar individually
        for yi, md, lo, hi, col_ in zip(y, sub['mean_delta'], sub['ci_low'], sub['ci_high'], colors_b):
            ax.errorbar(md, yi,
                        xerr=[[md - lo], [hi - md]],
                        fmt='o', color='black', ecolor=col_, elinewidth=3, capsize=4)

        ax.axvline(0, color='grey', ls='--', lw=1)
        ax.set_yticks(y); ax.set_yticklabels(sub['metric'])
        ax.set_xlabel('mean(JFDS − baseline)  with 95% bootstrap CI')
        ax.set_title(f'{model_name} — JFDS vs {baseline_method}', fontsize=10)

plt.suptitle('R8 — Effect Size with Uncertainty vs. Every Baseline (colored = CI excludes zero)', y=1.01, fontsize=12)
plt.tight_layout()
plt.savefig('r8_bootstrap_ci.png', dpi=110, bbox_inches='tight')
plt.show()


---
## R9 — Group Fairness by User Demographics  ✨ NEW

Break down NDCG, F (novelty), and D (ILD) by Gender, Age bucket, and Occupation.
Answers: *does JFDS improve recommendations equally across all user groups, or does it help some while hurting others?*


In [ ]:
# Merge user demographics into metrics_long
demo_cols = ['u_idx', 'Gender', 'Age', 'Occupation']
metrics_demo = metrics_long.merge(
    users_df[demo_cols].rename(columns={'u_idx': 'user'}),
    on='user', how='left'
)

# Age bucketing
age_map = {1: '<18', 18: '18-24', 25: '25-34', 35: '35-44', 45: '45-49', 50: '50-55', 56: '56+'}
metrics_demo['Age_label'] = metrics_demo['Age'].map(age_map).fillna('Unknown')

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
group_cols  = ['Gender', 'Age_label', 'Occupation']
group_titles = ['Gender', 'Age group', 'Occupation']
metric_pairs = [('ndcg', 'NDCG@10'), ('F', 'Fairness (novelty)'), ('D', 'Diversity (ILD)')]

for col_idx, (demo_col, demo_title) in enumerate(zip(group_cols, group_titles)):
    for row_idx, (metric, metric_label) in enumerate(metric_pairs):
        ax = axes[row_idx][col_idx]
        grp = (metrics_demo.groupby([demo_col, 'method', 'base_model'])[metric]
               .mean().reset_index())

        groups = sorted(grp[demo_col].unique())
        x      = np.arange(len(groups))
        width  = 0.2
        offsets = {'TopK_SVD': -1.5, 'JFDS_SVD': -0.5, 'TopK_BPR': 0.5, 'JFDS_BPR': 1.5}
        colors  = {'TopK_SVD': C_TOPK, 'JFDS_SVD': C_JFDS,
                   'TopK_BPR': '#85C1E9', 'JFDS_BPR': '#F1948A'}

        for method in ['TopK', 'JFDS']:
            for bm in ['SVD', 'BPR']:
                sub   = grp[(grp['method'] == method) & (grp['base_model'] == bm)]
                vals  = [sub[sub[demo_col] == g][metric].values[0]
                         if len(sub[sub[demo_col] == g]) > 0 else 0 for g in groups]
                key   = f'{method}_{bm}'
                ax.bar(x + offsets[key] * width, vals, width,
                       label=key, color=colors[key], alpha=0.85)

        ax.set_xticks(x)
        ax.set_xticklabels(groups, rotation=35, ha='right', fontsize=8)
        ax.set_ylabel(metric_label, fontsize=9)
        ax.set_title(f'{metric_label} by {demo_title}', fontsize=9)
        if row_idx == 0 and col_idx == 2:
            ax.legend(fontsize=7, loc='upper right')

plt.suptitle('R9 — Group Fairness: JFDS vs Top-K across User Demographics', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('r9_group_fairness.png', dpi=110, bbox_inches='tight')
plt.show()

# Statistical test: does JFDS help all groups equally? (Kruskal-Wallis on delta per demo group)
print("\nKruskal-Wallis test: is the JFDS gain in NDCG uniform across groups?")
for demo_col, demo_title in zip(group_cols, group_titles):
    for bm in ['SVD', 'BPR']:
        topk_d = metrics_demo[(metrics_demo.method=='TopK') & (metrics_demo.base_model==bm)].set_index('user')
        jfds_d = metrics_demo[(metrics_demo.method=='JFDS') & (metrics_demo.base_model==bm)].set_index('user')
        common = topk_d.index.intersection(jfds_d.index)
        delta_df = pd.DataFrame({
            'delta_ndcg': jfds_d.loc[common, 'ndcg'].values - topk_d.loc[common, 'ndcg'].values,
            demo_col: jfds_d.loc[common, demo_col].values,
        })
        groups_data = [g['delta_ndcg'].values for _, g in delta_df.groupby(demo_col) if len(g) > 5]
        if len(groups_data) >= 2:
            h_stat, p_val = stats.kruskal(*groups_data)
            print(f"  {bm} × {demo_title}: H={h_stat:.2f}  p={p_val:.4f}"
                  f"  {'← unequal gains' if p_val < 0.05 else '(uniform gains)'}")


---
## R10 — Aggregate Diversity & Exposure Lorenz Curve  ✨ NEW

Answers:
1. How many *distinct* items does each strategy recommend across all users? (Aggregate diversity)
2. How equitably is exposure distributed across items? (Lorenz curve + Gini)
3. Is exposure significantly different between Top-K and JFDS for head/mid/tail items? (Chi-square)


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 11))

agg_stats = []
for (model_name, method), rec_lists in lists.items():
    exp    = exposure_vector(rec_lists)
    agg_d  = aggregate_diversity(rec_lists)
    cov    = (exp > 0).sum() / N_ITEMS

    # Lorenz curve
    exp_sorted = np.sort(exp)
    cum_exp    = np.cumsum(exp_sorted) / exp_sorted.sum()
    cum_items  = np.linspace(0, 1, len(exp_sorted))
    exp_gini   = gini(exp)

    ax_idx = (0 if model_name == 'SVD' else 1, 0 if method == 'TopK' else 1)
    ax     = axes[ax_idx[0]][ax_idx[1]]
    ax.plot(cum_items, cum_exp, lw=2,
            color=C_TOPK if method == 'TopK' else C_JFDS,
            label=f'{model_name}-{method} (Gini={exp_gini:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', lw=0.8, label='Perfect equality')
    ax.fill_between(cum_items, cum_items, cum_exp, alpha=0.12,
                    color=C_TOPK if method == 'TopK' else C_JFDS)
    ax.set_xlabel('Fraction of items (sorted by exposure)')
    ax.set_ylabel('Fraction of total exposure')
    ax.set_title(f'{model_name} – {method}\nAggregate diversity: {agg_d:,} items  |  Coverage: {cov:.2%}',
                 fontsize=10)
    ax.legend(fontsize=9)

    # Tier-level exposure shares
    tier_exp = {t: exp[tier == t].sum() for t in TIERS}
    total_exp = sum(tier_exp.values())
    tier_shares = {t: tier_exp[t] / total_exp for t in TIERS}
    agg_stats.append({
        'base_model': model_name, 'method': method,
        'agg_diversity': agg_d, 'coverage': round(cov, 4), 'exposure_gini': round(exp_gini, 4),
        **{f'{t}_share': round(tier_shares[t], 4) for t in TIERS},
    })

plt.suptitle('R10 — Lorenz Curves of Item Exposure (area below diagonal = inequality)', fontsize=12)
plt.tight_layout()
plt.savefig('r10_lorenz_curves.png', dpi=110, bbox_inches='tight')
plt.show()

agg_df = pd.DataFrame(agg_stats).set_index(['base_model', 'method'])
display(agg_df)

# Chi-square test: are tier exposure shares significantly different JFDS vs TopK?
print("\nChi-square test: JFDS vs Top-K tier exposure distribution")
for bm in ['SVD', 'BPR']:
    exp_topk = exposure_vector(lists[(bm, 'TopK')])
    exp_jfds = exposure_vector(lists[(bm, 'JFDS')])

    obs_full = np.array([[exp_topk[tier == t].sum() for t in TIERS],
                          [exp_jfds[tier == t].sum()  for t in TIERS]])

    # chi2_contingency requires every column total > 0 (else expected freq = 0).
    # Drop tiers that got zero exposure from BOTH methods before testing.
    col_totals   = obs_full.sum(axis=0)
    keep_mask    = col_totals > 0
    dropped_tiers = [t for t, keep in zip(TIERS, keep_mask) if not keep]
    kept_tiers    = [t for t, keep in zip(TIERS, keep_mask) if keep]
    obs = obs_full[:, keep_mask]

    if dropped_tiers:
        print(f"  {bm}: dropping tier(s) with zero total exposure from the test: {dropped_tiers}")

    if obs.shape[1] < 2:
        print(f"  {bm}: not enough non-zero tiers remain ({obs.shape[1]}) to run chi-square test — skipping.")
        continue

    chi2, p, dof, _ = stats.chi2_contingency(obs)
    print(f"  {bm}: chi2={chi2:.2f}  p={p:.2e}  dof={dof}"
          f"  {'← significantly different' if p < 0.05 else '(not significant)'}")
    print(f"       TopK tier shares: { {t: round(obs[0,i]/obs[0].sum(),3) for i,t in enumerate(kept_tiers)} }")
    print(f"       JFDS tier shares: { {t: round(obs[1,i]/obs[1].sum(),3) for i,t in enumerate(kept_tiers)} }")

---
## R10b — Bootstrap Confidence Interval for Aggregate Diversity

Aggregate diversity (distinct items covered across all users) is a
**catalogue-level** statistic, not a per-user one — there's no meaningful
"per-user aggregate diversity" to pair the way R7/R8 pair per-user metrics.
Instead we bootstrap-resample the user population with replacement, recompute
aggregate diversity from each resample's union of recommendations, and read
off a 95% CI and an empirical one-sided p-value for JFDS vs. each baseline.


In [ ]:
# ── Bootstrap CI: Aggregate Diversity, JFDS vs each baseline ─────────────
N_BOOT_AGGDIV = 200  # kept modest — resampling the full ~6,000-user population is not free

def build_incidence_matrix(rec_lists_dict, n_items=N_ITEMS):
    """Boolean (n_users, n_items) matrix: which items each user's list contains."""
    users_sorted = sorted(rec_lists_dict.keys())
    mat = np.zeros((len(users_sorted), n_items), dtype=bool)
    for i, u in enumerate(users_sorted):
        mat[i, rec_lists_dict[u]] = True
    return users_sorted, mat

def bootstrap_aggdiv_diff(mat_base, mat_jfds, n_boot=N_BOOT_AGGDIV, seed=RANDOM_SEED):
    """Resample users with replacement; return array of (AggDiv_JFDS - AggDiv_base) / N_ITEMS per resample."""
    rng = np.random.default_rng(seed)
    n = mat_base.shape[0]
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        idx_b = rng.integers(0, n, n)
        agg_base = mat_base[idx_b].any(axis=0).sum()
        agg_jfds = mat_jfds[idx_b].any(axis=0).sum()
        diffs[b] = (agg_jfds - agg_base) / N_ITEMS
    return diffs

aggdiv_boot_rows = []
for model_name in ['SVD', 'BPR']:
    users_jfds, mat_jfds = build_incidence_matrix(lists[(model_name, 'JFDS')])
    for baseline_method in BASELINE_METHODS:
        users_base, mat_base = build_incidence_matrix(lists[(model_name, baseline_method)])
        # Align on the common, identically-ordered user set before resampling jointly.
        common_users = sorted(set(users_jfds) & set(users_base))
        idx_j = [users_jfds.index(u) for u in common_users]
        idx_b = [users_base.index(u) for u in common_users]
        diffs = bootstrap_aggdiv_diff(mat_base[idx_b], mat_jfds[idx_j])

        ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
        p_emp = max((diffs <= 0).mean(), 1.0 / N_BOOT_AGGDIV)

        aggdiv_boot_rows.append({
            'base_model': model_name, 'baseline': baseline_method,
            'mean_diff (coverage frac)': diffs.mean(),
            '95% CI lower': ci_lo, '95% CI upper': ci_hi,
            'Empirical p (one-sided)': p_emp,
            'CI excludes 0': not (ci_lo <= 0 <= ci_hi),
        })

aggdiv_boot_df = pd.DataFrame(aggdiv_boot_rows)
print(f'=== Bootstrap comparison: Aggregate Diversity, JFDS vs each baseline ({N_BOOT_AGGDIV} resamples) ===')
print(aggdiv_boot_df.to_string(index=False, float_format=lambda x: f'{x:.4g}'))


---
## v3 Additions — Adaptive-λ JScore, Sensitivity, Overlap & Multi-Resample Robustness

Everything below ports over what `jfds_experiment_v3.ipynb` tested that this notebook
didn't: an **adaptive λ-schedule** (λ_f, λ_d change with list position instead of
staying fixed for the whole run), a **sensitivity sweep** across schedule shapes,
**item-overlap@K** between Top-K and the new adaptive equation, and a **robustness
check across many independent user resamples** with box plots and sign-stability.

These cells reuse the SVD/BPR models, candidate pools, and metric helpers already
built above — no retraining required.

### A. Adaptive-λ JFDS Equation

In every cell above, λ_f and λ_d were **constants for an entire run** (the Optuna
search found one best static pair per base model). Here they instead **grow with
list position** — early slots (small *k*) stay utility-heavy, later slots shift
weight toward fairness/diversity:

$$t_k = \frac{k}{K-1}, \qquad
\lambda_f(k) = \lambda_f^{start} + (\lambda_f^{end}-\lambda_f^{start})\cdot t_k^{\,p}, \qquad
\lambda_d(k) = \lambda_d^{start} + (\lambda_d^{end}-\lambda_d^{start})\cdot t_k^{\,p}$$

with $\lambda_u(k) = 1-\lambda_f(k)-\lambda_d(k)$ recomputed every step. $p=1$ is
linear, $p>1$ back-loaded, $p<1$ front-loaded. Reuses the same `greedy_rerank`
engine, `fair_boost`, and `diversity_term_fast` as fixed-λ JFDS above.

In [ ]:
# ── Adaptive-λ schedule: λ_f, λ_d grow with list position instead of staying fixed ──
DEFAULT_SCHEDULE = dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=1.0)

def schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p=1.0):
    """t_k = step / (k_total - 1); lambda grows from *_start to *_end as t_k^p."""
    t = step / max(1, k_total - 1)
    lam_f = lf_start + (lf_end - lf_start) * (t ** p)
    lam_d = ld_start + (ld_end - ld_start) * (t ** p)
    return lam_f, lam_d

def adaptive_jfds_score(cand_idx, rel_value, selected, tier_counts, step, k_total=K,
                         lf_start=DEFAULT_SCHEDULE['lf_start'], lf_end=DEFAULT_SCHEDULE['lf_end'],
                         ld_start=DEFAULT_SCHEDULE['ld_start'], ld_end=DEFAULT_SCHEDULE['ld_end'],
                         p=DEFAULT_SCHEDULE['p']):
    lam_f, lam_d = schedule_lambdas(step, k_total, lf_start, lf_end, ld_start, ld_end, p)
    lam_u = 1 - lam_f - lam_d
    return (lam_u * rel_value
            + lam_f * fair_boost(cand_idx, tier_counts, step)
            + lam_d * diversity_term_fast(cand_idx, selected))

def adaptive_jfds_rerank(cand_ids, cand_rel, k=K, **schedule_params):
    params = {**DEFAULT_SCHEDULE, **schedule_params}
    return greedy_rerank(cand_ids, cand_rel, adaptive_jfds_score, k=k, k_total=k, **params)

print('Adaptive-λ JFDS equation defined ✓  —  λ_f, λ_d now grow with list position k, not fixed per run')
print('Default schedule:', DEFAULT_SCHEDULE)

### B. Generate Adaptive-λ JFDS Recommendation Lists

In [ ]:
print('Re-ranking all users with Adaptive-λ JFDS ...')
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    adaptive_lists = {}
    for u in range(N_USERS):
        cand_ids, cand_rel = pools[u]
        if len(cand_ids) == 0:
            continue
        adaptive_lists[u] = adaptive_jfds_rerank(cand_ids, cand_rel)
    lists[(model_name, 'AdaptiveJFDS')] = adaptive_lists
    print(f"  {model_name}: {len(adaptive_lists)} Adaptive-JFDS lists  (schedule={DEFAULT_SCHEDULE})")

### C. Extend Evaluation Metrics to Cover Adaptive-λ JFDS

In [ ]:
adaptive_rows = []
for model_name in ['SVD', 'BPR']:
    rec_lists = lists[(model_name, 'AdaptiveJFDS')]
    for u, rec in rec_lists.items():
        relevant_set = test_relevant.get(u, set())
        grades       = test_grades.get(u, {})
        p, r, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
        adaptive_rows.append({
            'user': u, 'base_model': model_name, 'method': 'AdaptiveJFDS',
            'precision': p, 'recall': r, 'ndcg': n_ndcg,
            'D': ild(rec), 'F': novelty_fairness(rec),
            'H': shannon_entropy(rec), 'gini_exp': gini(pop_count[rec]),
            'calibration_kl': calibration_kl(rec, train_seen.get(u, set())),
        })

adaptive_metrics_df = pd.DataFrame(adaptive_rows)
adaptive_metrics_df['U'] = adaptive_metrics_df['ndcg']
metrics_long = pd.concat([metrics_long, adaptive_metrics_df], ignore_index=True)
print(f"metrics_long now: {len(metrics_long):,} rows (added {len(adaptive_metrics_df):,} Adaptive-JFDS rows)")

# Headline comparison: Top-K vs fixed-λ JFDS vs Adaptive-λ JFDS
adaptive_summary = (metrics_long[metrics_long.method.isin(['TopK', 'JFDS', 'AdaptiveJFDS'])]
                     .groupby(['base_model', 'method'])[['precision', 'recall', 'ndcg', 'D', 'F', 'calibration_kl']]
                     .mean().round(4))
display(adaptive_summary)

### D. Item Overlap@K — Top-K vs Adaptive-λ JFDS

How many of the same movies does Adaptive-λ JFDS keep from the plain Top-K list,
per user? Low overlap means the equation is substituting, not just re-ordering.

In [ ]:
print('=== Item overlap@K: Top-K vs Adaptive-λ JFDS ===')
overlap_results = {}
for model_name in ['SVD', 'BPR']:
    topk_lists = lists[(model_name, 'TopK')]
    adap_lists = lists[(model_name, 'AdaptiveJFDS')]
    common_users = set(topk_lists) & set(adap_lists)
    overlaps = np.array([len(set(topk_lists[u]) & set(adap_lists[u])) for u in common_users])
    overlap_results[model_name] = overlaps
    print(f"  {model_name}: mean overlap@{K} = {overlaps.mean():.2f}  |  "
          f"median = {np.median(overlaps):.1f}  |  "
          f"users with 0 overlap = {(overlaps == 0).sum()} / {len(overlaps)} "
          f"({(overlaps == 0).mean() * 100:.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, model_name in zip(axes, ['SVD', 'BPR']):
    ax.hist(overlap_results[model_name], bins=np.arange(0, K + 2) - 0.5, color=C_JFDS, edgecolor='black', alpha=0.85)
    ax.set_xlabel(f'# items in common (Top-K ∩ Adaptive-JFDS), out of {K}')
    ax.set_ylabel('# users')
    ax.set_title(f'{model_name}: List overlap distribution')
    ax.set_xticks(range(0, K + 1))
plt.suptitle('Item Overlap@K — Top-K vs Adaptive-λ JFDS', y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig('overlap_topk_vs_adaptive.png', dpi=110, bbox_inches='tight')
plt.show()

### E. Sensitivity Check — Does the λ-Schedule Shape Matter?

Repeat the Adaptive-λ comparison for a few alternative schedules (different `p`
exponent, different start/end weights) on a fixed sample of users per base model,
to see whether conclusions depend heavily on the particular schedule chosen.

In [ ]:
ALT_SCHEDULES = {
    'linear (default)':      dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=1.0),
    'front-loaded (p=0.5)':  dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=0.5),
    'back-loaded (p=2.0)':   dict(lf_start=0.05, lf_end=0.45, ld_start=0.05, ld_end=0.35, p=2.0),
    'wider range (0.0→0.6)': dict(lf_start=0.00, lf_end=0.60, ld_start=0.00, ld_end=0.40, p=1.0),
}

N_USERS_SENS = min(300, N_USERS)
rng_sens = np.random.default_rng(RANDOM_SEED)

sens_rows = []
for model_name, pools in [('SVD', svd_pools), ('BPR', bpr_pools)]:
    sampled_u = rng_sens.choice(N_USERS, size=N_USERS_SENS, replace=False)
    for sched_name, sched_params in ALT_SCHEDULES.items():
        all_sel, niche_fracs, ilds, ndcgs = set(), [], [], []
        for u in sampled_u:
            cand_ids, cand_rel = pools[u]
            if len(cand_ids) == 0:
                continue
            rec = adaptive_jfds_rerank(cand_ids, cand_rel, **sched_params)
            all_sel.update(rec)
            niche_fracs.append(novelty_fairness(rec))
            ilds.append(ild(rec))
            relevant_set = test_relevant.get(u, set())
            grades = test_grades.get(u, {})
            _, _, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
            ndcgs.append(n_ndcg)
        fairness = float(np.mean(niche_fracs))
        agg_div  = len(all_sel) / N_ITEMS
        ild_mean = float(np.mean(ilds))
        jfds_composite = fairness * agg_div
        sens_rows.append({'base_model': model_name, 'Schedule': sched_name,
                          'Fairness': fairness, 'AggDiv': agg_div, 'ILD': ild_mean,
                          'JFDS_composite': jfds_composite, 'NDCG@K': float(np.mean(ndcgs))})
        print(f"[{model_name}] {sched_name:<26s} → Fairness={fairness:.4f}  AggDiv={agg_div:.4f}  "
              f"ILD={ild_mean:.4f}  JFDS={jfds_composite:.5f}  NDCG={np.mean(ndcgs):.4f}")

sens_df = pd.DataFrame(sens_rows)
display(sens_df.round(4))

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
metrics_sens = ['Fairness', 'AggDiv', 'ILD', 'JFDS_composite']
for row, model_name in enumerate(['SVD', 'BPR']):
    sub = sens_df[sens_df.base_model == model_name]
    for col, metric in enumerate(metrics_sens):
        ax = axes[row][col]
        bars = ax.bar(sub['Schedule'], sub[metric], color=plt.cm.Set2(np.linspace(0, 1, len(sub))), edgecolor='black')
        for b, v in zip(bars, sub[metric]):
            ax.text(b.get_x() + b.get_width() / 2, v, f'{v:.3f}', ha='center', va='bottom', fontsize=7)
        ax.set_title(f'{model_name}: {metric}', fontsize=9)
        ax.tick_params(axis='x', rotation=30, labelsize=7)
plt.suptitle('Sensitivity of Adaptive-λ JFDS metrics to schedule shape', y=1.03, fontsize=12)
plt.tight_layout()
plt.savefig('sensitivity_schedule_shape.png', dpi=110, bbox_inches='tight')
plt.show()

### F. Robustness — Many Independent User Resamples

All results above (for Adaptive-λ JFDS) were computed on the full evaluated user
population. To check they aren't an artifact of a lucky user mix, redraw **30
independent random subsamples** of users (no retraining needed — reranking is
already done for everyone) and see whether Adaptive-λ JFDS's ordering relative to
Top-K, MMR, and Fairness-only stays stable, the way `jfds_experiment_v3.ipynb` did
for its Strategy B.

In [ ]:
N_REPEATS = 30
N_RESAMPLE = min(300, N_USERS)
RERUN_SEED_BASE = 1000
strategy_labels = ['TopK', 'MMR', 'FairOnly', 'AdaptiveJFDS']

rerun_rows = []
for model_name in ['SVD', 'BPR']:
    for rep in range(N_REPEATS):
        seed = RERUN_SEED_BASE + rep
        rng_rep = np.random.default_rng(seed)
        sampled_u = rng_rep.choice(N_USERS, size=N_RESAMPLE, replace=False)
        for method in strategy_labels:
            rec_lists = lists[(model_name, method)]
            all_sel = set()
            f_vals, d_vals, ndcg_vals = [], [], []
            for u in sampled_u:
                rec = rec_lists.get(u)
                if not rec:
                    continue
                all_sel.update(rec)
                f_vals.append(novelty_fairness(rec))
                d_vals.append(ild(rec))
                relevant_set = test_relevant.get(u, set())
                grades = test_grades.get(u, {})
                _, _, n_ndcg = precision_recall_ndcg(rec, relevant_set, grades)
                ndcg_vals.append(n_ndcg)
            rerun_rows.append({
                'base_model': model_name, 'rep': rep, 'seed': seed, 'Strategy': method,
                'Fairness': np.mean(f_vals), 'AggDiv': len(all_sel) / N_ITEMS,
                'ILD': np.mean(d_vals), 'NDCG': np.mean(ndcg_vals),
            })

rerun_df = pd.DataFrame(rerun_rows)
print(f"Done — {len(rerun_df)} rows ({N_REPEATS} reruns × {len(strategy_labels)} strategies × 2 base models).")

rerun_summary = rerun_df.groupby(['base_model', 'Strategy'])[['Fairness', 'AggDiv', 'ILD', 'NDCG']].agg(['mean', 'std'])
display(rerun_summary.round(4))

In [ ]:
metrics_box = ['Fairness', 'AggDiv', 'ILD', 'NDCG']
colors_map = {'TopK': '#7F8C8D', 'MMR': '#2980B9', 'FairOnly': '#E67E22', 'AdaptiveJFDS': '#27AE60'}

for model_name in ['SVD', 'BPR']:
    sub_model = rerun_df[rerun_df.base_model == model_name]
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, metric in zip(axes, metrics_box):
        data = [sub_model.loc[sub_model.Strategy == s, metric].values for s in strategy_labels]
        bp = ax.boxplot(data, patch_artist=True, widths=0.6, medianprops=dict(color='black', linewidth=1.5))
        for patch, s in zip(bp['boxes'], strategy_labels):
            patch.set_facecolor(colors_map[s]); patch.set_alpha(0.75)
        rng_jit = np.random.default_rng(0)
        for i, s in enumerate(strategy_labels):
            vals = sub_model.loc[sub_model.Strategy == s, metric].values
            jitter = rng_jit.uniform(-0.12, 0.12, size=len(vals))
            ax.scatter(np.full(len(vals), i + 1) + jitter, vals, color='black', s=8, alpha=0.4, zorder=3)
        ax.set_title(metric, fontsize=11)
        ax.set_xticks(range(1, len(strategy_labels) + 1))
        ax.set_xticklabels(strategy_labels, rotation=20, ha='right')
    plt.suptitle(f'{model_name}: Distribution across {N_REPEATS} independent resamples of {N_RESAMPLE} users each',
                 y=1.08, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'resample_boxplots_{model_name.lower()}.png', dpi=110, bbox_inches='tight')
    plt.show()

In [ ]:
baseline_order = ['TopK', 'MMR', 'FairOnly']
baseline_colors = {'TopK': '#7F8C8D', 'MMR': '#2980B9', 'FairOnly': '#E67E22'}
sign_stability_rows = []

for model_name in ['SVD', 'BPR']:
    sub_model = rerun_df[rerun_df.base_model == model_name]
    wide = sub_model.pivot(index='rep', columns='Strategy', values=metrics_box)
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    for ax, metric in zip(axes, metrics_box):
        adap_vals = wide[metric]['AdaptiveJFDS'].values
        deltas = {b: adap_vals - wide[metric][b].values for b in baseline_order}
        bp = ax.boxplot([deltas[b] for b in baseline_order], patch_artist=True, widths=0.6,
                         medianprops=dict(color='black', linewidth=1.5))
        for patch, b in zip(bp['boxes'], baseline_order):
            patch.set_facecolor(baseline_colors[b]); patch.set_alpha(0.75)
        ax.axhline(0, color='red', linestyle='--', linewidth=1, alpha=0.7)
        ax.set_title(f'AdaptiveJFDS − baseline\n{metric}', fontsize=10)
        ax.set_xticks(range(1, len(baseline_order) + 1))
        ax.set_xticklabels(baseline_order, rotation=20, ha='right')
    plt.suptitle(f'{model_name}: Sign stability across {N_REPEATS} resamples (AdaptiveJFDS vs baselines)',
                 y=1.1, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'sign_stability_{model_name.lower()}.png', dpi=110, bbox_inches='tight')
    plt.show()

    print(f'=== {model_name}: fraction of {N_REPEATS} reruns where AdaptiveJFDS beats each baseline ===')
    for metric in metrics_box:
        adap_vals = wide[metric]['AdaptiveJFDS'].values
        for b in baseline_order:
            base_vals = wide[metric][b].values
            frac = (adap_vals > base_vals).mean()
            print(f'  {metric:10s} vs {b:10s}: AdaptiveJFDS higher in {frac * 100:5.1f}% of reruns')
            sign_stability_rows.append({'base_model': model_name, 'metric': metric, 'baseline': b, 'frac_adaptive_wins': frac})
    print()

sign_stability_df = pd.DataFrame(sign_stability_rows)
display(sign_stability_df)

### Summary of v3-Ported Sections

- **Adaptive-λ JFDS** (`AdaptiveJFDS`) is now a fifth strategy alongside Top-K,
  JFDS, MMR, and Fairness-only, generated for the **full** user population on
  both SVD and BPR — see `adaptive_summary` for the headline comparison.
- **`overlap_results`** holds per-user item-overlap@K counts (Top-K vs
  Adaptive-JFDS) for both base models.
- **`sens_df`** holds the schedule-shape sensitivity sweep (linear / front-loaded /
  back-loaded / wider-range) with Fairness, AggDiv, ILD, JFDS-composite, and NDCG.
- **`rerun_df`** / **`sign_stability_df`** hold the 30-resample robustness check —
  box plots show whether Adaptive-JFDS's ranking relative to each baseline is
  stable across independently drawn user subsamples, and the win-fraction table
  quantifies how often the sign ever flips.

---
## Conclusion

In [ ]:
n_sig_favor_jfds = int(
    (((sig_table['wilcoxon_p_holm'] < 0.05) & (sig_table['mean_delta'] > 0) & (sig_table['metric'] != 'calibration_kl')).sum()) +
    (((sig_table['wilcoxon_p_holm'] < 0.05) & (sig_table['mean_delta'] < 0) & (sig_table['metric'] == 'calibration_kl')).sum())
)
n_metrics_tested = len(sig_table)
n_aggdiv_sig = int(aggdiv_boot_df['CI excludes 0'].sum())

display(Markdown(f'''
Out of **{n_metrics_tested}** (metric × baseline × base-model) comparisons in R7 — now covering
**Top-K, MMR, and Fairness-only** as baselines, not just Top-K — JFDS is significantly better
(Wilcoxon p<0.05, **Holm–Bonferroni corrected** for all {n_metrics_tested} comparisons) on
**{n_sig_favor_jfds}** of them. See `sig_table` for the full per-comparison breakdown, including
which specific baseline/metric/model combinations do *not* favor JFDS (JFDS is not expected to
beat MMR on diversity or Fairness-only on raw fairness — each specialist baseline should be
strongest on the one thing it optimizes alone; the interesting claim is that JFDS wins on the
**joint** objective without being catastrophically behind either specialist on its own metric).

**Baseline comparison bars & delta table:** see `baseline_delta_df` for exactly how much JFDS
gains or gives up vs. Top-K, MMR, and Fairness-only on Fairness / Diversity / NDCG.

**R1 (Pareto):** see `pareto_stats` for % of Top-K lists dominated by JFDS on (F, D, U).

**R2 (Scaling law):** best-fitting F→D curve: **{best_name}** — `{scaling_table.iloc[0]["Equation"]}`.

**R4 (Conservation):** most stable combination: F^{best_alpha:.2f} × D^{best_beta:.2f} (CV={best_cv:.4f}).

**R7/R7b/R8:** `sig_table` (with Holm-corrected p-values) and `boot_df` hold exact p-values and
effect sizes per metric, per baseline, per model.

**R9 (Group fairness):** see bar charts and Kruskal-Wallis results — check whether JFDS gains are
uniform across gender, age, and occupation groups.

**R10/R10b (Aggregate diversity):** see `agg_df` for distinct-item counts and Lorenz Gini per
strategy, chi-square results for tier-exposure shifts, and `aggdiv_boot_df` for bootstrap
confidence intervals confirming (or not) that JFDS's aggregate-diversity advantage over each
baseline holds up beyond a single point estimate — **{n_aggdiv_sig} / {len(aggdiv_boot_df)}**
baseline comparisons have a 95% CI that excludes zero.

**Sanity check:** the single-user, four-strategy comparison earlier in the notebook ties these
aggregate numbers back to real, human-readable recommendation lists.

**To swap in a new JFDS equation:** edit only the cell marked "JFDS equation — EDIT THIS CELL".
Every cell after it re-runs unchanged — including the MMR and Fairness-only baselines, which
depend only on `greedy_rerank`, `GENRE_SIM`, and `tier`, not on the JFDS score itself.

**Adaptive-λ JFDS (v3-ported, see "v3 Additions" section above):** letting λ_f and λ_d grow
with list position (instead of staying fixed for the whole run) gives a fifth strategy,
`AdaptiveJFDS`, evaluated on the **full** user population for both SVD and BPR — see
`adaptive_summary` for headline numbers vs. Top-K and fixed-λ JFDS. Item-overlap@K
(`overlap_results`) shows how much the adaptive list actually substitutes rather than just
re-orders Top-K. The schedule-shape sensitivity sweep (`sens_df`) checks whether the linear vs.
front-loaded vs. back-loaded vs. wider-range ramp changes the conclusions. Finally, the
30-independent-resample robustness check (`rerun_df`, `sign_stability_df`) confirms whether
Adaptive-JFDS's ranking relative to Top-K / MMR / Fairness-only is a stable property of the
dataset or an artifact of one particular user sample — see the box plots and win-fraction table
for the full breakdown.
'''))
